In [1]:
import pandas as pd
from tab_forge.dataset import Dataset
from tab_forge.prompt_generator import PromptGenerator
from tab_forge.llm_runner import  LLMRunner

In [2]:
df = pd.read_csv("abalone.csv")

In [3]:
dataset = Dataset(
    df,
    target="Rings",
    task_type="regression",
    categorical_features=["Sex"],
    numerical_features=["Length", "Diameter", "Height", "Whole weight", "Shucked weight", "Viscera weight", "Shell weight"]
)

In [4]:
gen = PromptGenerator()

In [5]:
prompt = gen.build_prompt(
    dataset       = dataset,
    target_metric = "r2_metric",
    shot_mode     = "zero",    
    mfe_features  = "short"
)


[1/4] Loading statistics txt files...
  [OK] CTGAN: 0/0 datasets
  [OK] WGAN-GP: 0/0 datasets
  [OK] GAN-MFS: 0/0 datasets
  [OK] CTABGAN+: 0/0 datasets
  [OK] TVAE: 0/0 datasets
  [OK] DDPM: 0/0 datasets

[2/4] Extracting meta-features for 0 reference datasets...

[3/4] Extracting meta-features for user dataset...
    pymfe added 0 features (groups: ['general', 'statistical'])

[4/4] Generating prompt (mode: zero-shot)...


In [6]:
print(prompt)

════════════════════════════════════════════════════════════════════════
You are an expert in generative models for tabular data.

You are given:
  1. Meta-features of the user dataset

Your goal is to rank the 6 generative models listed below from most to least likely to perform well on the given dataset with respect to the provided target quality metric.

════════════════════════════════════════════════════════════════════════
## GENERATIVE MODELS

  CTGAN: Conditional GAN for tabular data using mode-specific normalization and conditional sampling. A well-established baseline that handles categorical imbalance through resampling, though performance can degrade on datasets with strong feature dependencies.
  WGAN-GP: Wasserstein GAN with gradient penalty enforcing Lipschitz continuity. Provides stable adversarial training with consistent convergence behavior across a range of tabular dataset types.
  GAN-MFS: WGAN-GP variant that adds a meta-feature distribution matching objective to 

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

True

In [8]:
api_key = os.getenv('LLM_API_KEY')
model = "openai/gpt-4o-mini"
base_url = "https://routerai.ru/api/v1"

In [9]:
runner = LLMRunner(
    base_url = base_url,
    api_key  = api_key,
    model    = model,
    retry_on_parse_error = True,   # перезапустить если блок не распарсился
    request_delay        = 1.0,    # пауза между запросами (для rate limit)
)

In [10]:
result = runner.run(
    prompt      = prompt,
    n_runs      = 3,
    temperature = 0.2,
    top_p       = 0.95,
    max_tokens  = 2048,
)

  run 1/3... OK
  run 2/3... OK
  run 3/3... OK


In [11]:
result.average_ranks

{'DDPM': 1.6666666666666667,
 'GAN-MFS': 1.6666666666666667,
 'WGAN-GP': 2.6666666666666665,
 'CTABGAN+': 4.333333333333333,
 'CTGAN': 5,
 'TVAE': 5.666666666666667}

In [12]:
result.final_ranking

['DDPM', 'GAN-MFS', 'WGAN-GP', 'CTABGAN+', 'CTGAN', 'TVAE']

In [14]:
result.runs[0].ranking

['DDPM', 'GAN-MFS', 'WGAN-GP', 'CTABGAN+', 'CTGAN', 'TVAE']

In [15]:
result

LLMRunner result  |  model=openai/gpt-4o-mini  |  runs=3  parsed=3/3
──────────────────────────────────────────────────
Average ranks (lower = better):
  1. DDPM          avg_rank=1.67
  2. GAN-MFS       avg_rank=1.67
  3. WGAN-GP       avg_rank=2.67
  4. CTABGAN+      avg_rank=4.33
  5. CTGAN         avg_rank=5.00
  6. TVAE          avg_rank=5.67
──────────────────────────────────────────────────
Individual runs:
  run  1  [OK]  DDPM > GAN-MFS > WGAN-GP > CTABGAN+ > CTGAN > TVAE
  run  2  [OK]  GAN-MFS > WGAN-GP > DDPM > CTGAN > CTABGAN+ > TVAE
  run  3  [OK]  DDPM > GAN-MFS > WGAN-GP > CTABGAN+ > TVAE > CTGAN